In [ ]:
from dotenv import find_dotenv, load_dotenv
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_pymupdf4llm import PyMuPDF4LLMLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
_ = load_dotenv(find_dotenv())

In [ ]:
# 1. Load PDF
loader = PyMuPDF4LLMLoader("./sample_doc.pdf")
docs = loader.load()
print(f"Loaded {len(docs)} pages from PDF")
print(f"Sample Content: {docs[0].page_content}")

In [ ]:
# 2. Chunking
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

splits = splitter.split_documents(docs)
print(f"Splits in {len(splits)} Chunks")

In [ ]:
# 3. Embedding
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [ ]:
# 4. Store
vector_store = FAISS.from_documents(splits, embedding=embeddings)

# Test Similarity Search
query = "چه اتفاقی برای کشتی سپیدار افتاده بود؟"
results = vector_store.similarity_search(query=query, k=2)
for i, doc in enumerate(results):
    print(f"{i+1}: {doc.page_content[:100]}")

In [ ]:
# 5. RAG with LCEL
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
فقط و فقط بر اساس Context جواب بده و اگه اطلاعاتی پیدا نکردی صراحتا و دوستانه بگو نمی دونم و به هیچ وجه از خودت جوابی نساز.

Context:
{context}
 """,
        ),
        ("human", "{question}"),
    ]
)

context = retriever | format_docs

chain = (
    {"context": context, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


while True:
    query = input("").strip()

    if query == "q":
        break
    print(f"شما: {query}")

    response = chain.invoke(query)

    print(f"دستیار: {response}")

### Semantic Chunking

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker

text = """
گوشی‌های سامسونگ از محبوب‌ترین گوشی‌های بازار هستند.
مدل گلکسی S24 دوربین بسیار باکیفیتی دارد.
باتری این گوشی برای استفاده روزمره مناسب است.
صفحه‌نمایش AMOLED رنگ‌ها را بسیار دقیق نمایش می‌دهد.
پردازنده این گوشی عملکرد خوبی در اجرای بازی‌ها دارد.

خودروهای BMW به عملکرد اسپرت معروف هستند.
مدل BMW M4 موتور قدرتمندی دارد.
شتاب این خودرو برای علاقه‌مندان به رانندگی جذاب است.
سیستم تعلیق BMW M4 برای رانندگی اسپرت طراحی شده است.
طراحی داخلی این خودرو ترکیبی از فناوری و امکانات لوکس است.

یادگیری ماشین یکی از شاخه‌های هوش مصنوعی است.
الگوریتم رگرسیون لجستیک برای مسائل طبقه‌بندی استفاده می‌شود.
درخت تصمیم داده‌ها را بر اساس ویژگی‌ها تقسیم می‌کند.
جنگل تصادفی از ترکیب چندین درخت تصمیم ساخته می‌شود.
شبکه‌های عصبی می‌توانند الگوهای پیچیده را یاد بگیرند.

ورزش منظم به حفظ سلامت بدن کمک می‌کند.
تمرینات قدرتی باعث افزایش قدرت عضلات می‌شوند.
پیاده‌روی روزانه می‌تواند سلامت قلب را بهبود دهد.
مصرف پروتئین کافی برای بازسازی عضلات اهمیت دارد.
خواب مناسب به ریکاوری بدن پس از تمرین کمک می‌کند.
"""
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
splitter = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=85,
)

splits = splitter.create_documents([text])

for i, doc in enumerate(splits, 1):
    print(f"\n{'=' * 20} Chunk {i} {'=' * 20}")
    print(doc.page_content)


==================== Chunk 1 ====================

گوشی‌های سامسونگ از محبوب‌ترین گوشی‌های بازار هستند. مدل گلکسی S24 دوربین بسیار باکیفیتی دارد. باتری این گوشی برای استفاده روزمره مناسب است.

==================== Chunk 2 ====================
صفحه‌نمایش AMOLED رنگ‌ها را بسیار دقیق نمایش می‌دهد. پردازنده این گوشی عملکرد خوبی در اجرای بازی‌ها دارد. خودروهای BMW به عملکرد اسپرت معروف هستند. مدل BMW M4 موتور قدرتمندی دارد. شتاب این خودرو برای علاقه‌مندان به رانندگی جذاب است. سیستم تعلیق BMW M4 برای رانندگی اسپرت طراحی شده است. طراحی داخلی این خودرو ترکیبی از فناوری و امکانات لوکس است. یادگیری ماشین یکی از شاخه‌های هوش مصنوعی است.

==================== Chunk 3 ====================
الگوریتم رگرسیون لجستیک برای مسائل طبقه‌بندی استفاده می‌شود. درخت تصمیم داده‌ها را بر اساس ویژگی‌ها تقسیم می‌کند. جنگل تصادفی از ترکیب چندین درخت تصمیم ساخته می‌شود. شبکه‌های عصبی می‌توانند الگوهای پیچیده را یاد بگیرند. ورزش منظم به حفظ سلامت بدن کمک می‌کند.

==================== Chunk 4 ====================
تمری